# 4 · Analysis & Figures

Load the BB84 results from the FABRIC run, visualize the photon pipeline, compare against
theory, cross-validate against QFabric's simulation, and break down the photon budget.

**Prerequisite:** `02_run_experiment` produced `results/fabric_*_results.json`.
The repo ships **sample results**, so this notebook also runs standalone for a demo.

### At a glance
- **Purpose:** turn the recorded results into the figures and tables (photon pipeline, theory comparison, loss budget, cross-platform bars).
- **Inputs:** `results/fabric_*_results.json` and (optional) `results/cross_validation.json` from notebook 3.
- **Outputs:** inline plots and summary tables.
- **Runs on / runtime:** anywhere (no slice); < 1 min. Ships with sample results so it works standalone.
- **If something fails:** if `cross_validation.json` is absent, Section 5 falls back to FABRIC-vs-local-sim and tells you to run notebook 3 for the full comparison.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from qne.metrics import ExperimentMetrics
from qne.config import ScenarioConfig
from validation.scenario import ValidationScenario
from validation.run_qfabric import run_qfabric_bb84_simulated

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load FABRIC Results

In [ ]:
alice = ExperimentMetrics.from_json(PROJECT_DIR / "results" / "fabric_alice_results.json")
bob = ExperimentMetrics.from_json(PROJECT_DIR / "results" / "fabric_bob_results.json")

# Alice has the authoritative sent count; Bob has the receiver perspective
metrics = alice
print(f"Loaded scenario: {metrics.scenario_name}")
print(f"Photons sent: {metrics.photons_sent:,}")
print(f"Photons received: {metrics.photons_received:,}")
print(f"Sifted bits: {metrics.sifted_bits:,}")
print(f"Final key bits: {metrics.final_key_bits:,}")
print(f"QBER: {metrics.qber:.4f}")
print(f"Elapsed: {metrics.elapsed_seconds:.1f}s")

## 2. Experiment Summary

In [ ]:
cfg = metrics.config
summary = pd.DataFrame([
    ["Scenario", metrics.scenario_name],
    ["Sites", "UCSD \u2194 STAR (FABRIC)"],
    ["Distance (km)", cfg["channel"]["distance_km"]],
    ["Attenuation (dB/km)", cfg["channel"]["attenuation_db_per_km"]],
    ["Detector efficiency", cfg["detector"]["efficiency"]],
    ["Dark count rate (Hz)", cfg["detector"]["dark_count_rate"]],
    ["Photons sent", f"{metrics.photons_sent:,}"],
    ["Sample fraction", cfg["protocol"]["sample_fraction"]],
    ["Seed", cfg.get("seed", "N/A")],
    ["Start time", metrics.start_time],
    ["End time", metrics.end_time],
    ["Elapsed (s)", f"{metrics.elapsed_seconds:.1f}"],
], columns=["Parameter", "Value"])

summary.style.hide(axis="index")

## 3. Photon Pipeline Waterfall

Track photon counts through each stage of the BB84 pipeline, with
theoretical expectations overlaid.

In [ ]:
# Reconstruct the pipeline from config
sc = ScenarioConfig.from_dict(cfg)
n_sent = metrics.photons_sent
channel_loss_prob = sc.loss_probability
det_eff = sc.detector.efficiency

# Theoretical expectations
theory_after_channel = n_sent * (1 - channel_loss_prob)
theory_detected = theory_after_channel * det_eff
theory_sifted = theory_detected * 0.5  # basis match probability
theory_final = theory_sifted * (1 - cfg["protocol"]["sample_fraction"])

# Actual counts
actual_after_channel = metrics.photons_received / det_eff  # back out detector effect
actual_detected = metrics.photons_received
actual_sifted = metrics.sifted_bits
actual_final = metrics.final_key_bits

stages = ["Sent", "Channel\nsurvivors", "Detected", "Sifted", "Final key"]
actual_vals = [n_sent, actual_after_channel, actual_detected, actual_sifted, actual_final]
theory_vals = [n_sent, theory_after_channel, theory_detected, theory_sifted, theory_final]

x = np.arange(len(stages))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, actual_vals, width=0.5, color="steelblue", alpha=0.85, label="FABRIC actual")
ax.plot(x, theory_vals, "o--", color="darkorange", linewidth=2, markersize=8, label="Theory")

for i, (a, t) in enumerate(zip(actual_vals, theory_vals)):
    ax.text(i, a + n_sent * 0.02, f"{a:,.0f}", ha="center", va="bottom", fontsize=9, color="steelblue")
    ax.text(i, t - n_sent * 0.05, f"{t:,.0f}", ha="center", va="top", fontsize=9, color="darkorange")

ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel("Photon / bit count")
ax.set_title("BB84 Photon Pipeline: FABRIC vs Theory")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## 4. Theory Comparison Table

Compare actual FABRIC results against theoretical predictions derived from
the scenario configuration.

In [ ]:
# Compute actual rates
actual_channel_loss = metrics.photons_lost / metrics.photons_sent
# Combined loss = channel + detector; separate detector contribution
actual_detection_rate = metrics.photons_received / metrics.photons_sent
actual_basis_match = metrics.sifted_bits / metrics.photons_received if metrics.photons_received else 0

# Expected values
expected_channel_loss = channel_loss_prob
expected_overall_detection = (1 - channel_loss_prob) * det_eff
expected_basis_match = 0.5

comparison = pd.DataFrame([
    ["Channel loss",
     f"{expected_channel_loss:.4f} ({expected_channel_loss*100:.2f}%)",
     f"{actual_channel_loss:.4f} ({actual_channel_loss*100:.2f}%)",
     f"{abs(actual_channel_loss - expected_channel_loss):.4f}"],
    ["Overall detection rate",
     f"{expected_overall_detection:.4f} ({expected_overall_detection*100:.2f}%)",
     f"{actual_detection_rate:.4f} ({actual_detection_rate*100:.2f}%)",
     f"{abs(actual_detection_rate - expected_overall_detection):.4f}"],
    ["Basis match rate",
     f"{expected_basis_match:.4f} ({expected_basis_match*100:.1f}%)",
     f"{actual_basis_match:.4f} ({actual_basis_match*100:.2f}%)",
     f"{abs(actual_basis_match - expected_basis_match):.4f}"],
    ["QBER",
     f"{(1 - cfg['channel']['polarization_fidelity'])/2:.4f} (intrinsic, (1-F)/2)",
     f"{metrics.qber:.4f} ({metrics.qber*100:.2f}%)",
     f"{metrics.qber:.4f}"],
    ["Secure key rate (bits/photon)",
     "-",
     f"{metrics.secure_key_rate:.4f}",
     "-"],
], columns=["Metric", "Expected", "Actual (FABRIC)", "|Delta|"])

comparison.style.hide(axis="index")

## 5. Cross-Platform Comparison

Compare QBER and secure key rate across **all backends** from the on-FABRIC
cross-validation (notebook 3): QFabric measured, QFabric-sim, SeQUeNCe, NetSquid.
If `results/cross_validation.json` isn't present (notebook 3 not run yet), this
falls back to FABRIC measured vs a local QFabric simulation.

In [ ]:
import json

xval_path = PROJECT_DIR / 'results' / 'cross_validation.json'
if xval_path.exists():
    backends = json.loads(xval_path.read_text())
    print(f'Loaded {len(backends)} backends from notebook 03 (on-FABRIC cross-validation).')
else:
    print('No cross_validation.json found — run notebook 03 for SeQUeNCe/NetSquid.')
    print('Falling back to FABRIC measured vs local QFabric-sim.')
    sim_scenario = ValidationScenario(
        name=metrics.scenario_name,
        distance_km=cfg['channel']['distance_km'],
        attenuation_db_per_km=cfg['channel']['attenuation_db_per_km'],
        detector_efficiency=cfg['detector']['efficiency'],
        dark_count_rate_hz=cfg['detector']['dark_count_rate'],
        polarization_fidelity=cfg['channel']['polarization_fidelity'],
        num_photons=metrics.photons_sent,
        sample_fraction=cfg['protocol']['sample_fraction'],
        seed=cfg.get('seed', 42),
    )
    sim = run_qfabric_bb84_simulated(sim_scenario)
    backends = [
        {'platform': 'qfabric', 'qber': metrics.qber, 'sifted_bits': metrics.sifted_bits,
         'secure_key_rate': metrics.secure_key_rate},
        {'platform': 'qfabric_sim', 'qber': sim.qber, 'sifted_bits': sim.sifted_bits,
         'secure_key_rate': sim.secure_key_rate},
    ]

# Order the platforms consistently for the plots/table.
_order = ['qfabric', 'qfabric_sim', 'sequence', 'netsquid']
present = ([b for p in _order for b in backends if b['platform'] == p]
           + [b for b in backends if b['platform'] not in _order])
for b in present:
    print(f"  {b['platform']:12s} QBER={b['qber']:.4f}  sifted={b.get('sifted_bits')}  "
          f"SKR={b.get('secure_key_rate', 0):.4f}")

In [ ]:
names = [b['platform'] for b in present]
qbers = [b['qber'] * 100 for b in present]
skrs  = [b.get('secure_key_rate', 0) for b in present]
palette = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'slategray']
colors = [palette[i % len(palette)] for i in range(len(names))]

fid = cfg['channel']['polarization_fidelity']
exp_qber = (1 - fid) / 2 * 100   # expected intrinsic QBER, percent

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.bar(names, qbers, color=colors)
ax.axhline(exp_qber, ls='--', color='gray', label=f'expected (1-F)/2 = {exp_qber:.2f}%')
for i, q in enumerate(qbers):
    ax.text(i, q, f'{q:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('QBER (%)'); ax.set_title('QBER by platform'); ax.legend()

ax = axes[1]
ax.bar(names, skrs, color=colors)
for i, s in enumerate(skrs):
    ax.text(i, s, f'{s:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Secure key rate (bits/photon)'); ax.set_title('Secure key rate by platform')

for ax in axes:
    ax.grid(axis='y', alpha=0.3); ax.tick_params(axis='x', rotation=15)
plt.tight_layout()

In [ ]:
cmp_df = pd.DataFrame([
    {'platform': b['platform'], 'QBER': round(b['qber'], 4),
     'sifted_bits': b.get('sifted_bits'),
     'secure_key_rate': round(b.get('secure_key_rate', 0), 4)}
    for b in present
])
cmp_df.style.hide(axis='index')

## 6. Loss Breakdown

Where were photons lost? Breakdown into channel loss, detector
inefficiency, basis mismatch, and QBER sampling overhead.

In [ ]:
# Decompose the photon budget
n_sent = metrics.photons_sent

# Channel survivors (before detector)
channel_survivors = metrics.photons_received / det_eff
lost_channel = n_sent - channel_survivors

# Lost at detector
lost_detector = channel_survivors - metrics.photons_received

# Lost to basis mismatch (detected but not sifted)
lost_basis = metrics.photons_received - metrics.sifted_bits

# Lost to QBER sampling (sifted bits used for estimation, not in final key)
lost_sampling = metrics.sifted_bits - metrics.final_key_bits

# Final key
final_key = metrics.final_key_bits

slices = [lost_channel, lost_detector, lost_basis, lost_sampling, final_key]
labels = [
    f"Channel loss\n({lost_channel:,.0f})",
    f"Detector inefficiency\n({lost_detector:,.0f})",
    f"Basis mismatch\n({lost_basis:,.0f})",
    f"QBER sampling\n({lost_sampling:,.0f})",
    f"Final key\n({final_key:,.0f})",
]
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#95a5a6", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    slices, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=140, pctdistance=0.8,
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title(f"Photon Budget Breakdown (n={n_sent:,} sent)")
plt.tight_layout()